# 06 — Validate organism mapping

## Goal

I keep the original ARMD organism label in the fact tables and apply reviewed mappings in a separate model. That gives me a source identity, a standardized identity, and a reporting group without losing the original terminology.

This notebook checks mapping coverage and, more importantly, looks for cases where several source labels from the same culture collapse to the same reporting organism.

In [1]:
%sql
select
    cohort_id,

    count(*) as culture_organism_rows,

    sum(
        case
            when mapping_status = 'reviewed' then 1
            else 0
        end
    ) as reviewed_mapping_rows,

    round(
        100.0
        * sum(
            case
                when mapping_status = 'reviewed' then 1
                else 0
            end
        )
        / count(*),
        1
    ) as reviewed_mapping_pct,

    count(distinct organism_source) as source_organism_labels,

    count(
        distinct case
            when mapping_status = 'reviewed'
                then organism_source
        end
    ) as reviewed_source_labels,

    count(
        distinct case
            when include_primary_summary = 1
                then organism_standardized
        end
    ) as primary_standardized_organisms

from workspace.micro_dev.int_mapped_cohort_culture_organism

group by cohort_id

order by cohort_id;

,cohort_id,culture_organism_rows,reviewed_mapping_rows,reviewed_mapping_pct,source_organism_labels,reviewed_source_labels,primary_standardized_organisms
0,blood,13651,9033,66.2,238,22,12
1,respiratory,22411,17491,78.0,167,20,12
2,urine,97184,81982,84.4,142,20,12


### Mapping coverage

The reviewed mapping set covers 66.2% of blood, 78.0% of respiratory, and 84.4% of urine culture-organism rows. I am not trying to map all 314 labels just to get a high coverage percentage; the primary reporting organisms are reviewed explicitly.

In [2]:
%sql
select
    cohort_id,
    organism_source,
    count(*) as culture_organism_rows,
    count(distinct patient_key) as patients

from workspace.micro_dev.int_mapped_cohort_culture_organism

where mapping_status = 'unmapped'

group by
    cohort_id,
    organism_source

order by
    cohort_id,
    culture_organism_rows desc;

,cohort_id,organism_source,culture_organism_rows,patients
0,blood,COAG NEGATIVE STAPHYLOCOCCUS,1913,1247
1,blood,STREPTOCOCCUS MITIS GROUP,262,251
2,blood,STREPTOCOCCUS ANGINOSUS GROUP,132,129
3,blood,STREPTOCOCCUS PYOGENES (GROUP A),122,120
4,blood,BACTEROIDES FRAGILIS,96,96
5,blood,VIRIDANS GROUP STREPTOCOCCI,86,83
6,blood,STREPTOCOCCUS DYSGALACTIAE,84,81
7,blood,ENTEROBACTER CLOACAE,63,62
8,blood,CITROBACTER FREUNDII COMPLEX,59,56
9,blood,STREPTOCOCCUS MITIS/STREPTOCOCCUS ORALIS,59,57


### Unmapped high-volume labels

This is my backlog for future terminology work. Leaving a label unmapped is preferable to assigning a biological identity that I cannot support.

In [3]:
%sql
select
    cohort_id,
    culture_key,
    organism_standardized,

    count(distinct organism_source) as source_label_count,

    array_sort(
        collect_set(organism_source)
    ) as source_labels

from workspace.micro_dev.int_mapped_cohort_culture_organism

where include_primary_summary = 1

group by
    cohort_id,
    culture_key,
    organism_standardized

having count(distinct organism_source) > 1

order by
    source_label_count desc,
    cohort_id,
    organism_standardized;

,cohort_id,culture_key,organism_standardized,source_label_count,source_labels
0,respiratory,1fb458768ff7a678418489c64ad5b67b,Staphylococcus aureus,3,"[STAPH AUREUS {MRSA}, STAPH AUREUS(COLONY VARIANT - SMALL COLONY OR OTHER MORPHOTYPE), STAPHYLOCOCCUS AUREUS]"
1,blood,d92862448e0fec67728ae7f30a0bc7af,Enterococcus faecium,2,"[ENTEROCOCCUS FAECIUM, ENTEROCOCCUS FAECIUM - VANCO RESISTANT]"
2,blood,c671c06a5e71de36b8393b7282dd96d1,Escherichia coli,2,"[ESCHERICHIA COLI, ESCHERICHIA COLI (CARBAPENEM RESISTANT)]"
3,blood,7e1c3012df06f03eccee35e98f086cb2,Staphylococcus aureus,2,"[STAPH AUREUS(COLONY VARIANT - SMALL COLONY OR OTHER MORPHOTYPE), STAPHYLOCOCCUS AUREUS]"
4,blood,a49634d47eae39e92da83daef8d5d1c4,Staphylococcus aureus,2,"[STAPH AUREUS {MRSA}, STAPHYLOCOCCUS AUREUS]"
5,blood,a22fd81033245e47e47b0aac5a5125cc,Staphylococcus aureus,2,"[STAPH AUREUS {MRSA}, STAPHYLOCOCCUS AUREUS]"
6,blood,6811eac316f0b8e528a80c41a56e8e15,Staphylococcus aureus,2,"[STAPH AUREUS {MRSA}, STAPH AUREUS(COLONY VARIANT - SMALL COLONY OR OTHER MORPHOTYPE)]"
7,blood,c4fa8eb94355c019f9079f6804ac2263,Staphylococcus aureus,2,"[STAPH AUREUS {MRSA}, STAPHYLOCOCCUS AUREUS]"
8,blood,979280069e73610941ba700d3acc4c6a,Staphylococcus aureus,2,"[STAPH AUREUS(COLONY VARIANT - SMALL COLONY OR OTHER MORPHOTYPE), STAPHYLOCOCCUS AUREUS]"
9,blood,6021cc9118c63001a2b31a1325995dcf,Staphylococcus aureus,2,"[STAPH AUREUS {MRSA}, STAPH AUREUS(COLONY VARIANT - SMALL COLONY OR OTHER MORPHOTYPE)]"


### Same-culture mapping collisions

This turned out to be an important finding. A culture can contain multiple source labels that map to the same base organism, for example ordinary *S. aureus* plus an MRSA-qualified label, or multiple *P. aeruginosa* morphotypes.

Those records should not be collapsed before resistance phenotypes are derived, because the qualifier can carry information that needs to be reconciled with AST.

In [4]:
%sql
with collisions as (

    select
        cohort_id,
        culture_key,
        organism_standardized

    from workspace.micro_dev.int_mapped_cohort_culture_organism

    where include_primary_summary = 1

    group by
        cohort_id,
        culture_key,
        organism_standardized

    having count(distinct organism_source) > 1

)

select
    cohort_id,
    organism_standardized,
    count(*) as culture_orders_with_multiple_source_labels

from collisions

group by
    cohort_id,
    organism_standardized

order by
    culture_orders_with_multiple_source_labels desc,
    cohort_id,
    organism_standardized;

,cohort_id,organism_standardized,culture_orders_with_multiple_source_labels
0,respiratory,Pseudomonas aeruginosa,1520
1,respiratory,Staphylococcus aureus,200
2,urine,Pseudomonas aeruginosa,52
3,blood,Staphylococcus aureus,9
4,urine,Enterobacter cloacae complex,3
5,urine,Klebsiella pneumoniae,2
6,blood,Enterococcus faecium,1
7,blood,Escherichia coli,1
8,respiratory,Enterobacter cloacae complex,1
9,urine,Escherichia coli,1


### Collision summary

Most of the collision volume is in respiratory *P. aeruginosa*, with smaller groups in *S. aureus* and several Enterobacterales mappings. This is what pushed the pipeline toward phenotype assignment first and culture-level consolidation second.

In [ ]:
%sql
select
    organism_source,
    organism_standardized,
    include_primary_summary,
    mapping_status

from workspace.micro_dev.int_mapped_cohort_culture_organism

where include_primary_summary = 1
  and organism_standardized is null;

,organism_source,organism_standardized,include_primary_summary,mapping_status


### Primary mapping completeness

This should return no rows. Anything marked for the primary summary must have a reviewed standardized identity.

In [3]:
%sql
with expected as (

    select *
    from values

        (
            'ENTEROBACTER ASBURIAE',
            'Enterobacter asburiae',
            'Enterobacter cloacae complex',
            null
        ),

        (
            'ENTEROBACTER CLOACAE',
            'Enterobacter cloacae',
            'Enterobacter cloacae complex',
            null
        ),

        (
            'KLEBSIELLA AEROGENES',
            'Klebsiella aerogenes',
            'Klebsiella aerogenes',
            null
        ),

        (
            'KLEBSIELLA AEROGENES (CARBAPENEM RESISTANT)',
            'Klebsiella aerogenes',
            'Klebsiella aerogenes',
            'carbapenem_resistant'
        ),

        (
            'ZZZENTEROBACTER AEROGENES',
            'Klebsiella aerogenes',
            'Klebsiella aerogenes',
            null
        ),

        (
            'CITROBACTER BRAAKII',
            'Citrobacter braakii',
            'Citrobacter freundii complex',
            null
        ),

        (
            'ZZZCITROBACTER FARMERI',
            'Citrobacter farmeri',
            'Citrobacter farmeri',
            null
        ),

        (
            'CITROBACTER FREUNDII',
            'Citrobacter freundii',
            'Citrobacter freundii complex',
            null
        ),

        (
            'CITROBACTER FREUNDII COMPLEX',
            'Citrobacter freundii complex',
            'Citrobacter freundii complex',
            null
        ),

        (
            'CITROBACTER FREUNDII COMPLEX (CARBAPENEM RESISTANT)',
            'Citrobacter freundii complex',
            'Citrobacter freundii complex',
            'carbapenem_resistant'
        ),

        (
            'CITROBACTER KOSERI',
            'Citrobacter koseri',
            'Citrobacter koseri',
            null
        ),

        (
            'CITROBACTER KOSERI (CARBAPENEM RESISTANT)',
            'Citrobacter koseri',
            'Citrobacter koseri',
            'carbapenem_resistant'
        ),

        (
            'CITROBACTER YOUNGAE',
            'Citrobacter youngae',
            'Citrobacter freundii complex',
            null
        )

    as expected(
        organism_source,
        expected_standardized,
        expected_reporting_group,
        expected_source_qualifier
    )

),

actual as (

    select
        organism_source,
        organism_standardized,
        organism_reporting_group,
        source_qualifier,
        mapping_status,

        count(*) as culture_organism_records,

        count(
            distinct patient_key
        ) as patients

    from workspace.micro_dev.int_mapped_cohort_culture_organism

    group by
        organism_source,
        organism_standardized,
        organism_reporting_group,
        source_qualifier,
        mapping_status

)

select
    e.organism_source,

    e.expected_standardized,
    a.organism_standardized as actual_standardized,

    e.expected_reporting_group,
    a.organism_reporting_group as actual_reporting_group,

    e.expected_source_qualifier,
    a.source_qualifier as actual_source_qualifier,

    a.mapping_status,
    a.culture_organism_records,
    a.patients,

    case
        when a.organism_source is null
            then 'missing_from_model'

        when a.organism_standardized
                 <> e.expected_standardized
            then 'standardized_identity_mismatch'

        when a.organism_reporting_group
                 <> e.expected_reporting_group
            then 'reporting_group_mismatch'

        when not (
            a.source_qualifier
                <=> e.expected_source_qualifier
        )
            then 'source_qualifier_mismatch'

        when a.mapping_status <> 'reviewed'
            then 'mapping_status_mismatch'

        else 'pass'
    end as validation_status

from expected e

left join actual a
    on e.organism_source = a.organism_source

order by
    e.expected_reporting_group,
    e.organism_source

,organism_source,expected_standardized,actual_standardized,expected_reporting_group,actual_reporting_group,expected_source_qualifier,actual_source_qualifier,mapping_status,culture_organism_records,patients,validation_status
0,ZZZCITROBACTER FARMERI,Citrobacter farmeri,Citrobacter farmeri,Citrobacter farmeri,Citrobacter farmeri,None,None,reviewed,8,8,pass
1,CITROBACTER BRAAKII,Citrobacter braakii,Citrobacter braakii,Citrobacter freundii complex,Citrobacter freundii complex,None,None,reviewed,31,30,pass
2,CITROBACTER FREUNDII,Citrobacter freundii,Citrobacter freundii,Citrobacter freundii complex,Citrobacter freundii complex,None,None,reviewed,202,181,pass
3,CITROBACTER FREUNDII COMPLEX,Citrobacter freundii complex,Citrobacter freundii complex,Citrobacter freundii complex,Citrobacter freundii complex,None,None,reviewed,806,701,pass
4,CITROBACTER FREUNDII COMPLEX (CARBAPENEM RESISTANT),Citrobacter freundii complex,Citrobacter freundii complex,Citrobacter freundii complex,Citrobacter freundii complex,carbapenem_resistant,carbapenem_resistant,reviewed,86,78,pass
5,CITROBACTER YOUNGAE,Citrobacter youngae,Citrobacter youngae,Citrobacter freundii complex,Citrobacter freundii complex,None,None,reviewed,17,17,pass
6,CITROBACTER KOSERI,Citrobacter koseri,Citrobacter koseri,Citrobacter koseri,Citrobacter koseri,None,None,reviewed,1060,871,pass
7,CITROBACTER KOSERI (CARBAPENEM RESISTANT),Citrobacter koseri,Citrobacter koseri,Citrobacter koseri,Citrobacter koseri,carbapenem_resistant,carbapenem_resistant,reviewed,331,290,pass
8,ENTEROBACTER ASBURIAE,Enterobacter asburiae,Enterobacter asburiae,Enterobacter cloacae complex,Enterobacter cloacae complex,None,None,reviewed,7,7,pass
9,ENTEROBACTER CLOACAE,Enterobacter cloacae,Enterobacter cloacae,Enterobacter cloacae complex,Enterobacter cloacae complex,None,None,reviewed,380,346,pass


### Reporting-group spot check

I added `organism_reporting_group` so taxonomy and reporting can be handled separately. This check verifies the reviewed Citrobacter, Enterobacter, and Klebsiella mappings against the values I intended.

The tested mappings pass, including keeping *Citrobacter koseri* and *C. farmeri* separate while grouping the reviewed *C. freundii* complex members together.

### Takeaway

At this point I have a reviewed organism layer, but I still have source-level records. The next step is to profile the antibiotic vocabulary and decide which resistance phenotypes can actually be derived from the available AST.